# COMP8410 A1 - Week 5: TRELLIS.2 vs Hunyuan3D 2.1 (single image to 3D) - Colab, A100
**Run the numbered cells in order. Every code cell starts with a `# [n.n]` label that says what it does and when to run it.**

| Section | What | When |
|---|---|---|
| 1 | Setup: Drive, repo, packages | every fresh runtime (1.2 instead after a kernel restart) |
| 2 | Build the two model environments | fresh runtime only. Skip if `/content/mamba/envs` already has `trellis2` and `hunyuan` (then run only 2.3 once) |
| 3 | Hugging Face login, access check, input photos | every session |
| 4 | Smoke tests (1 item, 1 seed) | before any full run |
| 5 | Full run: seed 0 first, seeds 1-2 only if budget allows | after smoke passes |
| 6 | Report, cost log, free the GPU | at the end |

Before saving: tick *Omit code cell output when saving*. **Disconnect the runtime when finished (cell 6.3): the A100 bills while idle.**
If the kernel crashes or restarts: run **1.2**, then **3.1**. Never re-run 1.1 (it deletes the repo clone).

## 1. Setup

In [ ]:
# [1.1] SETUP, fresh runtime only: mount Drive, clone the repo, install packages for scoring and background removal (~3 min)
%cd /content
import os
os.environ['MAMBA_ROOT_PREFIX'] = '/content/mamba'   # micromamba finds its environments here; must be set before any `micromamba run`
os.environ['HF_HOME'] = '/content/hf'                # must be set before huggingface_hub is imported
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs', 'w5_data', 'w5_envs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!pip -q install -r requirements.txt
!pip -q install trimesh scipy tabulate rembg onnxruntime transformers lpips
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# [1.2] RESTORE after a kernel crash or restart (use INSTEAD of 1.1; files on the VM are untouched, only Python variables are lost)
import os
os.environ['MAMBA_ROOT_PREFIX'] = '/content/mamba'
os.environ['HF_HOME'] = '/content/hf'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE = '/content/drive/MyDrive/comp8410_a1'
%cd "/content/a1/48576158 Evaluation"
from google.colab import drive
drive.mount('/content/drive')   # no-op if still mounted
!git -C /content/a1 pull -q
print('restored; now run 3.1 to log in to Hugging Face again')

In [ ]:
# [1.3] LINK DRIVE FOLDERS (results/ and outputs/ live on Drive) AND RUN THE CPU TESTS (fresh runtime, once, after 1.1)
%cd "/content/a1/48576158 Evaluation"
!rm -rf results outputs
!ln -s {BASE}/results results
!ln -s {BASE}/outputs outputs
!mkdir -p {BASE}/outputs/w5_figures
!python -m pytest -q tests/test_w5.py

## 2. Model environments (fresh runtime only)
Two separate environments, because the models need different torch builds. **Already built in your current session? Skip 2.1, 2.2 and 2.4-2.5 and run only 2.3 (the transformers pin) once.**
Building on a fresh runtime takes a long time on the A100 clock; 2.4 packs both environments to Drive (about 10-20 GB) so a later runtime can unpack them (2.5) instead.

In [ ]:
# [2.1] BUILD THE TRELLIS.2 ENVIRONMENT (micromamba, torch 2.6, CUDA 12.4 toolkit, flash-attn, o_voxel and the other extensions). Slow.
%cd "/content/a1/48576158 Evaluation"
!bash pairs/w5_image_to_3d/install_envs.sh micromamba
!bash pairs/w5_image_to_3d/install_envs.sh trellis2 2>&1 | tee {BASE}/logs/w5_install_trellis2.log | tail -40
# flash-attn is installed from a prebuilt wheel because the source build in setup.sh fails
!/content/bin/micromamba run -n trellis2 pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" 2>&1 | tail -2

In [ ]:
# [2.2] BUILD THE HUNYUAN3D 2.1 ENVIRONMENT (torch 2.5.1, custom rasterizer, mesh painter, bpy stub). The licence text prints at the end: read it.
%cd "/content/a1/48576158 Evaluation"
!bash pairs/w5_image_to_3d/install_envs.sh hunyuan 2>&1 | tee {BASE}/logs/w5_install_hunyuan.log | tail -60

In [ ]:
# [2.3] PIN transformers INSIDE THE TRELLIS.2 ENVIRONMENT (run once). setup.sh installs 5.x, where DINOv3ViTModel has no .layer (TRELLIS.2 reads it).
# Expected output: 'pin 4.57.1 -> 4.57.1 has .layer: True' (or 4.56.2). If both say False, stop and report it.
import subprocess
MT = ['/content/bin/micromamba', 'run', '-n', 'trellis2']
chk = ("import sys, transformers; from transformers import DINOv3ViTModel, AutoConfig; "
       "c = AutoConfig.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m'); m = DINOv3ViTModel(c); "
       "print(transformers.__version__, 'has .layer:', hasattr(m, 'layer')); sys.exit(0 if hasattr(m, 'layer') else 1)")
for v in ['4.57.1', '4.56.2']:
    subprocess.run(MT + ['pip', 'install', '-q', f'transformers=={v}'], capture_output=True, text=True)
    r = subprocess.run(MT + ['python', '-c', chk], capture_output=True, text=True)
    print('pin', v, '->', r.stdout.strip() or r.stderr[-300:])
    if r.returncode == 0:
        break

In [ ]:
# [2.4] OPTIONAL: PACK BOTH ENVIRONMENTS TO DRIVE so a later runtime can skip 2.1-2.2 (needs about 10-20 GB free; run after 2.3 works)
%cd "/content/a1/48576158 Evaluation"
!du -sh /content/mamba /content/TRELLIS.2 /content/Hunyuan3D-2.1
!bash pairs/w5_image_to_3d/install_envs.sh pack {BASE}/w5_envs

In [ ]:
# [2.5] OPTIONAL: UNPACK the environments from Drive (use INSTEAD of 2.1-2.3 on a new runtime when 2.4 was done earlier)
%cd "/content/a1/48576158 Evaluation"
!bash pairs/w5_image_to_3d/install_envs.sh unpack {BASE}/w5_envs

## 3. Hugging Face and input photos (every session)

In [ ]:
# [3.1] HUGGING FACE LOGIN (paste a read token when asked; it is not saved in the notebook). The account must have access to DINOv3 and RMBG-2.0.
import getpass, os
from huggingface_hub import login
tok = getpass.getpass('HF token: ')
login(token=tok)
os.environ['HF_TOKEN'] = tok

In [ ]:
# [3.2] ACCESS CHECK (run once): both gated repos must print OK with no FAIL line after them
import re
from huggingface_hub import hf_hub_download, HfApi
txt = open(hf_hub_download('microsoft/TRELLIS.2-4B', 'pipeline.json')).read()
repos = sorted(r for r in set(re.findall(r'"([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)"', txt)) if not r.startswith('ckpts/'))  # ckpts/... are folders inside the repo
print('gated repos named in pipeline.json:', repos)
api = HfApi()
for r in repos:
    try:
        api.auth_check(r); print('OK   ', r)
    except Exception as e:
        print('FAIL ', r, '->', str(e)[:100], ' (open the model page, accept the terms, then re-run 3.1)')

In [ ]:
# [3.3] INPUTS: clean copies of the 13 Set C photos, background removal (both models get the same cut-out), contact sheet. LOOK at the sheet.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w5_image_to_3d/prepare_inputs.py --raw {BASE}/w5_data/setC
!python pairs/w5_image_to_3d/run_w5.py prep 2>&1 | tail -20
from PIL import Image
display(Image.open('outputs/w5_figures/inputs_sheet.jpg'))

## 4. Smoke tests: 1 item (c01_topiary), 1 seed per model. Results go to separate smoke files.
Order: 4.1 and 4.2 generate, 4.3 scores them and shows the view sheet. Check the object is upright, textured, and resembles the photo. If a model lies on its side, change `UP_AXIS` in `w5_common.py`.

In [ ]:
# [4.1] SMOKE: TRELLIS.2 generates one mesh (first run downloads the weights, several minutes)
%cd "/content/a1/48576158 Evaluation"
!cd /content/TRELLIS.2 && /content/bin/micromamba run -n trellis2 python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_trellis2.py" --items c01_topiary --seeds 0 --out-root /content/smoke_out --results "/content/a1/48576158 Evaluation/results/w5_smoke.jsonl" 2>&1 | tee {BASE}/logs/w5_smoke_trellis2.log | grep -v Warning | tail -40

In [ ]:
# [4.2] SMOKE: Hunyuan3D 2.1 generates one mesh (about 3.5 min per mesh; the warm-up call is skipped)
%cd "/content/a1/48576158 Evaluation"
!cd /content/Hunyuan3D-2.1 && /content/bin/micromamba run -n hunyuan python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_hunyuan3d.py" --items c01_topiary --seeds 0 --no-warmup --out-root /content/smoke_out --results "/content/a1/48576158 Evaluation/results/w5_smoke.jsonl" 2>&1 | tee {BASE}/logs/w5_smoke_hunyuan.log | grep -v Warning | tail -30

In [ ]:
# [4.3] SMOKE: score both meshes, print SMOKE PASSED or FAILED, show the view sheets (photo first, then 8 rendered views)
# 'python -u' prints the [score] stage lines live, with memory use, so a crash shows which stage caused it
%cd "/content/a1/48576158 Evaluation"
!python -u pairs/w5_image_to_3d/score_w5.py --items c01_topiary --seeds 0 --out-root /content/smoke_out --results results/w5_smoke.jsonl --views-root /content/smoke_views 2>&1 | tail -40
import glob
from PIL import Image
from a1eval import load
rows = load('results/w5_smoke.jsonl')
need = {(m, k) for m in ('trellis2', 'hunyuan3d') for k in ('gen_ok', 'latency_s', 'dino_best', 'sil_iou_best', 'n_faces')}
got = {(r.model, r.metric) for r in rows if (r.metric != 'gen_ok' or r.value == 1.0)}
print('SMOKE PASSED' if need <= got else f'SMOKE FAILED, missing {sorted(need - got)}')
for f in sorted(glob.glob('/content/smoke_views/*/*.jpg')):
    print(f); display(Image.open(f))

## 5. Full run (13 items x 3 seeds x 2 models)
Stage 5.1-5.3 = seed 0 only (26 meshes, roughly 1 hour in total). Stages 5.4-5.5 add seeds 1 and 2 only if the budget allows. All generation cells are resumable: finished meshes are skipped, so after an interruption just run the same cell again. Meshes are written to `outputs/w5/` on Drive.

In [ ]:
# [5.1] FULL RUN, seed 0: TRELLIS.2 on all 13 items
%cd "/content/a1/48576158 Evaluation"
!cd /content/TRELLIS.2 && /content/bin/micromamba run -n trellis2 python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_trellis2.py" --seeds 0 2>&1 | tee -a {BASE}/logs/w5_gen_trellis2.log | grep -v Warning | tail -50

In [ ]:
# [5.2] FULL RUN, seed 0: Hunyuan3D 2.1 on all 13 items (about 3.5 min per mesh)
%cd "/content/a1/48576158 Evaluation"
!cd /content/Hunyuan3D-2.1 && /content/bin/micromamba run -n hunyuan python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_hunyuan3d.py" --seeds 0 --no-warmup 2>&1 | tee -a {BASE}/logs/w5_gen_hunyuan.log | grep -v Warning | tail -50

In [ ]:
# [5.3] SCORE every mesh generated so far (run after 5.2, and again after 5.4-5.5; already scored meshes are skipped)
%cd "/content/a1/48576158 Evaluation"
!python -u pairs/w5_image_to_3d/score_w5.py --models trellis2 hunyuan3d --seeds 0 1 2 2>&1 | tee {BASE}/logs/w5_score.log | tail -30

In [ ]:
# [5.4] OPTIONAL, only if budget allows: TRELLIS.2 seeds 1 and 2
%cd "/content/a1/48576158 Evaluation"
!cd /content/TRELLIS.2 && /content/bin/micromamba run -n trellis2 python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_trellis2.py" --seeds 1 2 2>&1 | tee -a {BASE}/logs/w5_gen_trellis2.log | grep -v Warning | tail -50

In [ ]:
# [5.5] OPTIONAL, only if budget allows: Hunyuan3D 2.1 seeds 1 and 2 (then re-run 5.3)
%cd "/content/a1/48576158 Evaluation"
!cd /content/Hunyuan3D-2.1 && /content/bin/micromamba run -n hunyuan python "/content/a1/48576158 Evaluation/pairs/w5_image_to_3d/gen_hunyuan3d.py" --seeds 1 2 --no-warmup 2>&1 | tee -a {BASE}/logs/w5_gen_hunyuan.log | grep -v Warning | tail -50

## 6. Results, cost, shut down

In [ ]:
# [6.1] TABLES, CHARTS AND QUALITATIVE GRID; copies report assets, library versions, repo commits and the Hunyuan licence to Drive
# The grid shows the photographed objects, so keep 'Omit code cell output when saving' ticked.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w5_image_to_3d/report_w5.py --show 2>&1 | tee {BASE}/logs/w5_report.log | grep -v Warning
!mkdir -p {BASE}/report_assets && cp -r report_assets/w5 {BASE}/report_assets/
!/content/bin/micromamba run -n trellis2 pip freeze > {BASE}/results/w5_ENV_trellis2.txt
!/content/bin/micromamba run -n hunyuan pip freeze > {BASE}/results/w5_ENV_hunyuan.txt
!git -C /content/TRELLIS.2 rev-parse HEAD > {BASE}/results/w5_commit_trellis2.txt
!git -C /content/Hunyuan3D-2.1 rev-parse HEAD > {BASE}/results/w5_commit_hunyuan.txt
!cp /content/Hunyuan3D-2.1/LICENSE* {BASE}/results/ 2>/dev/null
from PIL import Image
display(Image.open('outputs/w5_figures/w5_qualitative.jpg'))

In [ ]:
# [6.2] COST LOG: fill in HOURS (time the GPU runtime was connected) and UNITS_PER_HR (Resources panel), run once per session
%cd "/content/a1/48576158 Evaluation"
HOURS = None          # e.g. 4.5
UNITS_PER_HR = None   # e.g. 11.77 for the A100
if HOURS and UNITS_PER_HR:
    from a1eval import log_cost
    log_cost('cost_log.csv', '2026-10-31', 'Colab Pro', 'A100-40GB', HOURS, UNITS_PER_HR * 0.1513, note='W5 install + generation + scoring')
    !tail -3 cost_log.csv

In [1]:
# [6.3] FREE THE GPU: set AUTO_DISCONNECT = True once everything is copied to Drive (the A100 bills while idle)
AUTO_DISCONNECT = True
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()